# 20 — Value functions and the Bellman backup

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/020-value-functions-bellman.ipynb)

A value function answers one question for every state: how much reward is left if I play well from here? The Bellman equation turns that question into arithmetic you can repeat. In this notebook you run value iteration on a five-state corridor, watch the numbers climb into place, and then break it by bootstrapping off a terminal state.

## 1. The numbers: a corridor with one prize

Five states in a row, and stepping right is the only action. Entering the last state pays 1 and ends the episode, so every state's value is just that prize, discounted by the number of steps still to go. One backup fills in a single number; the rest of the notebook fills in the table.

In [ ]:
import numpy as np                              # the array library, preinstalled on Colab
reward = np.array([0., 0., 0., 0., 1.])         # reward paid on arriving in each state: only the goal pays
gamma = 0.9                                     # discount on future value
V = np.zeros(5)                                 # five states, all worth nothing at the start
print("reward paid on arrival:", reward)        # only the last state pays
print("V at the start:", V)                     # the table before any backup
print("V[3] after one backup:", reward[4] + gamma * V[4])   # 1.0: the prize, not yet discounted

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `reward = ...` — the reward paid when you arrive in each state; only the goal pays.
- `gamma = 0.9` — a reward one step later is worth 90% of one now.
- `V = np.zeros(5)` — the value table, initialised to zero everywhere.
- `print(...)` — the table before learning, then a single hand backup for state 3.

## 2. Step by step: one sweep at a time

A sweep replaces each state's value with the reward plus the discounted value of its neighbour. Because the new table is built from the old one, information travels only one state per sweep — the 1 appears at state 3, then state 2, and so on to the left.

In [ ]:
V = np.zeros(5)                                 # reset the table to zeros
for sweep in range(1, 9):                       # eight sweeps is enough to see the pattern
    V_new = V.copy()                            # build the next table from the current one
    for s in range(4):                          # states 0..3 have a neighbour to the right
        V_new[s] = reward[s + 1] + gamma * V[s + 1]   # reward for arriving, plus discounted value next
    V_new[4] = 0.0                              # the last state is terminal: no future value there
    V = V_new                                   # commit the whole sweep at once
    print(f"sweep {sweep}:", np.round(V, 3))    # one line of numbers per sweep

**Line by line**

- `V = np.zeros(5)` — start again from a fresh table.
- `for sweep in range(1, 9)` — each sweep is one pass over the states.
- `V_new = V.copy()` — a copy, so the sweep uses old values only.
- `V_new[s] = reward[s + 1] + gamma * V[s + 1]` — the reward for arriving, plus discounted value next.
- `V_new[4] = 0.0` — the terminal state keeps value zero forever.
- `V = V_new` — commit the sweep.
- `print(f"sweep {sweep}:", ...)` — watch one new state light up per sweep.

## 3. Plot: the values climbing into place

Run thirty sweeps and keep the whole table each time. Every state climbs towards its true value, and the dashed lines show where each one lands: 1, 0.9, 0.81, 0.729, then zero for the terminal state.

In [ ]:
import matplotlib.pyplot as plt                # plotting library, preinstalled on Colab

V = np.zeros(5)                                 # start from zeros again
history = [V.copy()]                            # the table after every sweep
for sweep in range(30):                         # thirty sweeps: plenty to converge
    V_new = V.copy()                            # next table from the current one
    for s in range(4):                          # the four states that have a successor
        V_new[s] = reward[s + 1] + gamma * V[s + 1]   # one Bellman backup
    V_new[4] = 0.0                              # terminal state stays at zero
    V = V_new                                   # commit the sweep
    history.append(V.copy())                    # remember it for the plot
history = np.array(history)                     # shape (31, 5): sweeps by states

plt.figure(figsize=(6, 3))                      # a wide, short figure
for s in range(4):                              # one line per state
    plt.plot(history[:, s], label=f"state {s}")  # value of state s against sweep number
    plt.axhline(gamma ** (3 - s), color="grey", linestyle="--", linewidth=0.8)   # the true value
plt.xlabel("sweep")                             # x-axis label
plt.ylabel("V(s)")                              # y-axis label
plt.legend(loc="lower right")                   # name the four lines
plt.title("each sweep copies the prize one state to the left")   # the takeaway
plt.show()                                      # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — the plotting library, aliased to `plt`.
- `history = [V.copy()]` — keep the starting table as sweep zero.
- `for sweep in range(30)` — thirty sweeps, far more than the corridor needs.
- `V_new[s] = ...` — the same backup as section 2.
- `history.append(V.copy())` — store a copy, or every entry would alias the same array.
- `np.array(history)` — turn the list into a (sweeps, states) grid.
- `plt.plot(history[:, s])` — column `s` is one state's history.
- `gamma ** (3 - s)` — the closed-form value: the prize discounted by the steps remaining.

## 4. Interactive: drag gamma

The discount factor sets the price of distance. Drag it towards 1 and the far states keep almost the whole prize; drag it towards 0 and only the state next to the prize is worth anything. The title reports the starting state's value.

In [ ]:
from ipywidgets import interact               # sliders that rerun a function

def show(gamma=0.9):                            # gamma comes from the slider
    Vg = np.zeros(5)                            # fresh table for this gamma
    for _ in range(80):                         # 80 sweeps is well past convergence
        V_next = Vg.copy()                      # next table from the current one
        for s in range(4):                      # the four non-terminal states
            V_next[s] = reward[s + 1] + gamma * Vg[s + 1]   # one Bellman backup
        V_next[4] = 0.0                         # terminal state
        Vg = V_next                             # commit
    plt.figure(figsize=(6, 3))                  # fresh figure for each drag
    plt.bar(np.arange(5), Vg, color="tab:blue")  # one bar per state
    plt.ylim(0, 1.05)                           # fixed axis, so nothing jumps
    plt.xlabel("state")                         # x-axis label
    plt.ylabel("V(s), value of being here")     # y-axis label
    plt.title(f"gamma={gamma:.2f} -> V(0)={Vg[0]:.3f}")   # live readout
    plt.show()                                  # draw it

interact(show, gamma=(0.0, 0.99, 0.01))         # the slider: range and step

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(gamma=0.9)` — the function that redraws one frame.
- `Vg = np.zeros(5)` — every drag starts learning from scratch.
- `for _ in range(80)` — repeated sweeps until the table stops moving.
- `V_next[s] = ...` — the same backup, now with the slider's gamma.
- `plt.bar(...)` — the converged table as five bars.
- `plt.title(f"...")` — the readout, so the numbers are not guesswork.
- `interact(show, gamma=(0.0, 0.99, 0.01))` — attach the slider and its step size.

## 5. When it breaks: bootstrapping off a terminal state

The last state is an exit, not a place: it has no future and must stay at zero. If the code forgets that and lets the terminal state bootstrap off itself, it pays its own reward forever and settles at 1/(1-gamma) — ten times too high here. Every value behind it is inflated by the same mistake.

In [ ]:
V_bad = np.zeros(5)                            # a second table, for the buggy version
for _ in range(80):                             # same number of sweeps as the slider
    V_next = np.zeros(5)                        # next table from the current one
    for s in range(5):                          # the bug: all five states are updated
        nxt = min(s + 1, 4)                     # the last state points at itself
        V_next[s] = reward[nxt] + gamma * V_bad[nxt]   # so it bootstraps off its own value
    V_bad = V_next                              # commit the sweep

V_good = np.zeros(5)                            # the correct table, for comparison
for _ in range(80):                             # same budget of sweeps
    V_next = V_good.copy()                      # copy, not a fresh array: state 4 must stay put
    for s in range(4):                          # only states that really have a successor
        V_next[s] = reward[s + 1] + gamma * V_good[s + 1]   # correct backup
    V_next[4] = 0.0                             # the exit is worth nothing
    V_good = V_next                             # commit

print("correct:", np.round(V_good, 3))          # 0.729 0.81 0.9 1.0 0.0
print("buggy:  ", np.round(V_bad, 3))           # everything inflated by about ten
print("terminal value, buggy:", round(V_bad[4], 3), "vs exact 1/(1-gamma) =", round(1 / (1 - gamma), 1))   # 10.0

plt.figure(figsize=(6, 3))                      # compare the two tables
plt.bar(np.arange(5) - 0.2, V_good, width=0.4, color="tab:green", label="terminal handled right")   # correct
plt.bar(np.arange(5) + 0.2, V_bad, width=0.4, color="tab:red", label="terminal bootstrapped")      # buggy
plt.xlabel("state")                             # x-axis label
plt.ylabel("V(s) after 80 sweeps")              # y-axis label
plt.legend(loc="upper left")                    # name the two bar sets
plt.title("one updatable terminal state inflates every value tenfold")   # the takeaway
plt.show()                                      # render

**Line by line**

- `V_bad = np.zeros(5)` — a second table for the buggy run.
- `for s in range(5)` — the bug in one character: the terminal state is updated too.
- `nxt = min(s + 1, 4)` — the last state's successor is itself.
- `V_next[s] = reward[nxt] + gamma * V_bad[nxt]` — so it collects the prize and its own future, forever.
- `V_next = V_good.copy()` — the correct run copies, so state 4 stays at zero.
- `print(...)` — the correct and buggy tables side by side.
- `plt.bar(...)` — paired bars, correct in green and buggy in red.

## Your turn

Set `gamma = 0.5` in section 1 and predict the correct value of state 0 before rerunning. Then explain why the buggy terminal value in section 5 would become 2 instead of 10 — and why the gap between the two tables shrinks.